# Bi-Int Twin — Evaluation Notebook

**Last updated:** 31 May 2026  
**Environment:** TwinCell (conda), Python 3.11, RDKit 2026.03, TensorFlow 2.15  
**Hardware used for training:** NVIDIA RTX 4000 Ada (20 GB VRAM), CUDA 13.0

---

## Contents

| Section | Description |
|---------|-------------|
| 1 | Setup & imports |
| 2 | CCLE dataset summary |
| 3 | ChEMBL GNN pre-training curves |
| 4 | Bi-Int QSAR — random split training curves |
| 5 | Bi-Int QSAR — Leave-Drug-Out results |
| 6 | Baseline comparison (all models × 3 splits) |
| 7 | GraphGA candidates — structures & drug-likeness |
| 8 | BRICS-DQN reward progression |
| 9 | Chemical validation (Tanimoto, SA, MedChem filters) |
| 10 | Conclusions & limitations |

> **⚠️ IC50 disclaimer:** Predicted IC50 values for de novo generated molecules
> are extrapolated by a model with limited leave-drug-out performance (r = 0.316).
> They must NOT be interpreted as reliable potency predictions.
> In vitro validation is required.

---
## 1. Setup & imports

All paths are relative to the `notebooks/` directory. The cell checks which
data files are available and prints a status for each.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import os, sys, warnings
warnings.filterwarnings('ignore')

# Make src/ importable (Bi-Int model classes)
ROOT = os.path.abspath('..')
for _p in [ROOT, os.path.join(ROOT, 'src')]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

try:
    from rdkit import Chem
    from rdkit.Chem import Draw, QED, Descriptors, AllChem, rdMolDescriptors, rdDepictor
    from rdkit.Chem.Draw import rdMolDraw2D
    from rdkit import RDLogger
    RDLogger.DisableLog('rdApp.*')
    from PIL import Image
    import io
    RDKIT_OK = True
except ImportError:
    print('RDKit not available — structure cells will be skipped')
    RDKIT_OK = False

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 10

# ── Data paths (relative to notebooks/) ──────────────────────────────────────
LOG_RANDOM   = os.path.join(ROOT, 'logs/run_gpu_main/training_log.csv')
LOG_LDO      = os.path.join(ROOT, 'logs/run_ldo/training_log.csv')
GA_CSV       = os.path.join(ROOT, 'graphga_top_candidates.csv')
DQN_CSV      = os.path.join(ROOT, 'Dataset/brics_dqn_results.csv')
BL_CSV       = os.path.join(ROOT, 'Dataset/baseline_results_with_CI.csv')
SMILES_CSV   = os.path.join(ROOT, 'Dataset/ccle_drug_smiles.csv')
VALID_CSV    = os.path.join(ROOT, 'Dataset/molecular_validation_report.csv')
TANIMOTO_CSV = os.path.join(ROOT, 'Dataset/graphga_tanimoto_vs_ccle.csv')
FIG_DIR      = os.path.join(ROOT, 'figures')

print('Setup OK')
for label, p in [
    ('Random log',  LOG_RANDOM),
    ('LDO log',     LOG_LDO),
    ('GA CSV',      GA_CSV),
    ('DQN CSV',     DQN_CSV),
    ('Baselines',   BL_CSV),
    ('SMILES map',  SMILES_CSV),
    ('Validation',  VALID_CSV),
    ('Tanimoto',    TANIMOTO_CSV),
]:
    status = '✓' if os.path.exists(p) else '✗ NOT FOUND'
    print(f'  {label:<15} {status}')

---
## 2. CCLE dataset summary

The CCLE (Cancer Cell Line Encyclopedia) 2019 dataset provides:
- IC50 measurements for 266 drugs across 1,068 cell lines
- Gene expression (978 landmark genes), copy-number alteration (426 genes),
  and mutation (735 genes) profiles for 647 common cell lines

Key preprocessing decisions:
1. **SMILES:** 201/266 drugs mapped via PubChem REST API (3-level lookup)
2. **IC50 transform:** `log1p(µM)`, then z-score per cell line
3. **Subsampling:** 20,000 triplets (seed 42) due to RAM/GPU constraints on full 103k

In [ ]:
ccle_summary = {
    'Drugs (IC50 matrix)':              266,
    'Drugs with valid SMILES':          201,
    'Drugs missing SMILES':              65,
    'Cell lines (GEx ∩ CNA ∩ IC50)':   647,
    'Valid triplets (drug, cell, IC50)': 103477,
    'Subsampled for training (seed 42)': 20000,
    'GEx dims':                          978,
    'CNA dims':                          426,
    'Mutation dims':                     735,
    'Mutation sparsity':                 0.844,
    'IC50 log1p mean (after zscore)':    2.67,
}
df_ccle = pd.DataFrame(list(ccle_summary.items()), columns=['Metric', 'Value'])
print(df_ccle.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
fig.suptitle('CCLE dataset overview', fontweight='bold')

axes[0].pie([201, 65], labels=['SMILES found (201)', 'Missing (65)'],
            colors=['#4CAF50', '#F44336'], autopct='%1.1f%%', startangle=90)
axes[0].set_title('Drug SMILES coverage (266 CCLE drugs)')

axes[1].bar(['GEx\n(978)', 'CNA\n(426)', 'Mut\n(735)'], [978, 426, 735],
            color=['#2196F3', '#FF9800', '#9C27B0'], edgecolor='k', linewidth=0.6)
axes[1].set_ylabel('Feature dimensions')
axes[1].set_title('Omics features per cell line (647 cell lines)')

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'nb_01_ccle_summary.png'), dpi=110, bbox_inches='tight')
plt.show()
print('Figure saved → figures/nb_01_ccle_summary.png')

---
## 3. ChEMBL GNN pre-training

The GNN drug encoder is pre-trained on 100k ChEMBL molecules via **self-supervised
multi-task regression**. The 8 regression targets (LogP, TPSA, MW, QED, HBD, HBA,
NumRings, NumAromaticRings) are computed by RDKit — no experimental labels needed.

This initialises the GNN with chemically meaningful representations before QSAR
fine-tuning on CCLE IC50.

**Best checkpoint:** epoch 9, val RMSE = 0.2187 (RMSE = 1.0 = predicting the mean).

In [ ]:
# Real results from 21 May 2026 run
epochs_pt  = list(range(1, 11))
train_rmse = [0.4886, 0.3462, 0.3034, 0.2796, 0.2649, 0.2511, 0.2410, 0.2322, 0.2247, 0.2192]
val_rmse   = [0.3598, 0.2972, 0.2671, 0.2471, 0.2467, 0.2327, 0.2323, 0.2300, 0.2187, 0.2215]
val_mae    = [0.2537, 0.2094, 0.1871, 0.1749, 0.1733, 0.1659, 0.1672, 0.1654, 0.1552, 0.1609]
best_ep    = int(np.argmin(val_rmse)) + 1  # epoch 9

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('ChEMBL GNN pre-training — 10 epochs (21 May 2026)', fontweight='bold')

axes[0].plot(epochs_pt, train_rmse, 'o-', label='Train RMSE', color='#2196F3', lw=2)
axes[0].plot(epochs_pt, val_rmse,   's-', label='Val RMSE',   color='#F44336', lw=2)
axes[0].plot(epochs_pt, val_mae,    '^--', label='Val MAE',   color='#FF9800', alpha=0.8)
axes[0].axvline(best_ep, color='green', linestyle='--', lw=1.5,
                label=f'Best checkpoint (ep {best_ep}, RMSE={min(val_rmse):.4f})')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('RMSE / MAE (normalised targets)')
axes[0].set_title('Learning curves\n(RMSE=1.0 = predicting global mean)')
axes[0].legend(fontsize=8); axes[0].set_xticks(epochs_pt); axes[0].grid(alpha=0.3)

val_loss = [0.12948, 0.08831, 0.07136, 0.06104, 0.06084, 0.05415, 0.05397, 0.05291, 0.04784, 0.04890]
axes[1].plot(epochs_pt, val_loss, 'o-', color='#9C27B0', lw=2)
axes[1].axvline(best_ep, color='green', linestyle='--', lw=1.5, label=f'Best (ep {best_ep})')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Val loss (MSE, 8 targets)')
axes[1].set_title('Validation loss convergence')
axes[1].legend(fontsize=8); axes[1].set_xticks(epochs_pt); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'nb_02_chembl_pretrain.png'), dpi=110, bbox_inches='tight')
plt.show()
print(f'Best epoch: {best_ep} — val RMSE = {min(val_rmse):.4f}')
print('Interpretation: val RMSE 0.22 = avg prediction error of 0.22σ across 8 descriptors')
print('Transferred layers: node_embed, gcn_proj_1/2, ln1/2, node_proj')
print('Figure saved → figures/nb_02_chembl_pretrain.png')

---
## 4. Bi-Int QSAR — random split (4 epochs)

**Configuration:** `src/fullPipeline.py --loss-mode cross_entropy --split-mode random`  
**Data:** 20k subsampled triplets, 201 drugs with real SMILES, 647 cell lines  

**Key finding:** Pearson r = **0.811** at epoch 4. However, this result is inflated
because the same drugs appear in train and validation — the model partially memorises
drug identity rather than learning to generalise.

| Epoch | Train RMSE | Val RMSE | Pearson r | Grad ‖∇‖ |
|-------|-----------|---------|-----------|----------|
| 1 | 0.9595 | 0.8542 | 0.506 | 26.15 |
| 2 | 0.7674 | 0.8986 | 0.631 | 13.39 |
| 3 | 0.6693 | 0.5936 | 0.791 | 11.12 |
| **4** | **0.6058** | **0.5881** | **0.811** | 9.40 |

Gradient norm fell 64% (26→9.4), indicating stable convergence. Epoch 5 triggered
GPU OOM (SelectV2 XLA, 17.6/20.5 GB VRAM).

In [ ]:
# Load from CSV if available; otherwise use hardcoded real values
qsar_fallback = {
    'epoch':      [1,      2,      3,      4],
    'train_rmse': [0.9595, 0.7674, 0.6693, 0.6058],
    'val_rmse':   [0.8542, 0.8986, 0.5936, 0.5881],
    'pearson_r':  [0.506,  0.631,  0.791,  0.811],
    'grad_norm':  [26.15,  13.39,  11.12,  9.40],
    'kl_loss':    [0.476,  0.456,  0.454,  0.452],
}
if os.path.exists(LOG_RANDOM):
    df_q = pd.read_csv(LOG_RANDOM)
    print(f'Loaded from {LOG_RANDOM}')
else:
    df_q = pd.DataFrame(qsar_fallback)
    print('Using hardcoded values (logs/ not present)')
print(df_q.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Bi-Int QSAR — Random Split (real data, 24 May 2026)', fontweight='bold')

axes[0].plot(df_q['epoch'], df_q['train_rmse'], 'o-', label='Train', color='#2196F3', lw=2)
axes[0].plot(df_q['epoch'], df_q['val_rmse'],   's-', label='Val',   color='#F44336', lw=2)
for ep, tr, va in zip(df_q['epoch'], df_q['train_rmse'], df_q['val_rmse']):
    axes[0].annotate(f'{tr:.3f}', (ep, tr), textcoords='offset points',
                     xytext=(0, 6), fontsize=7, ha='center', color='#2196F3')
    axes[0].annotate(f'{va:.3f}', (ep, va), textcoords='offset points',
                     xytext=(0,-12), fontsize=7, ha='center', color='#F44336')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('RMSE (normalised log µM)')
axes[0].set_title('RMSE — random split\n(RMSE=1.0 → predicting the mean)')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(df_q['epoch'], df_q['pearson_r'], 'o-', color='#4CAF50', lw=2, markersize=7)
axes[1].axhline(0.7, color='orange', linestyle='--', lw=1.2, label='Published CCLE floor (~0.70)')
for ep, r in zip(df_q['epoch'], df_q['pearson_r']):
    axes[1].annotate(f'{r:.3f}', (ep, r), textcoords='offset points', xytext=(0,6),
                     fontsize=8, ha='center')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Pearson r')
axes[1].set_title('Pearson r — validation\n(r=0.811, but optimistic — see LDO section)')
axes[1].set_ylim(0, 1); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

axes[2].bar(df_q['epoch'], df_q['grad_norm'],
            color=['#FF7043','#FFA726','#FFCA28','#D4E157'], edgecolor='k', lw=0.6)
for ep, g in zip(df_q['epoch'], df_q['grad_norm']):
    axes[2].text(ep, g+0.3, f'{g:.1f}', ha='center', fontsize=8)
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Gradient L2 norm')
axes[2].set_title('Gradient norm — 26.15→9.40 (−64%)\nStable convergence')
axes[2].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'nb_03_qsar_random.png'), dpi=110, bbox_inches='tight')
plt.savefig(os.path.join(FIG_DIR, '02_training_curves.png'), dpi=110, bbox_inches='tight')
plt.show()
print('Saved → figures/nb_03_qsar_random.png + figures/02_training_curves.png')

---
## 5. Bi-Int QSAR — Leave-Drug-Out (LDO)

**Split:** 171 train drugs | 30 unseen val drugs → ~16,832 / ~3,168 triplets  

This is the **honest generalisation metric**: the model must predict responses
to structurally novel drugs never seen during training.

**Key findings:**
- Best r = **0.316** at epoch 2 (statistically significant, p << 0.001)
- Overfitting starts at epoch 3: val RMSE jumps 0.983 → 1.158 (+17.7%)
- **XGBoost r = 0.367** (classical baseline, no deep learning) beats Bi-Int on LDO
- Root causes: insufficient training data (16k triplets), no early stopping, 9.2M params

In [ ]:
ldo_fallback = {
    'epoch':      [1,      2,      3,      4],
    'train_rmse': [0.9461, 0.7463, 0.6465, 0.6177],
    'val_rmse':   [0.9978, 0.9834, 1.1579, 1.1441],
    'pearson_r':  [0.253,  0.316,  0.209,  0.257],
    'grad_norm':  [32.11,  13.21,  10.60,  9.79],
}
if os.path.exists(LOG_LDO):
    df_ldo = pd.read_csv(LOG_LDO)
    print(f'Loaded from {LOG_LDO}')
else:
    df_ldo = pd.DataFrame(ldo_fallback)
    print('Using hardcoded values (logs/ not present)')
print(df_ldo.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Bi-Int QSAR — Leave-Drug-Out (real, 24 May 2026)', fontweight='bold')

axes[0].plot(df_ldo['epoch'], df_ldo['train_rmse'], 'o-', label='Train', color='#2196F3', lw=2)
axes[0].plot(df_ldo['epoch'], df_ldo['val_rmse'],   's-', label='Val',   color='#F44336', lw=2)
axes[0].axvspan(2.5, 4.5, alpha=0.08, color='red', label='Overfitting zone')
axes[0].axvline(2, color='green', linestyle='--', lw=1.5, label='Best checkpoint (ep 2, r=0.316)')
for ep, tr, va in zip(df_ldo['epoch'], df_ldo['train_rmse'], df_ldo['val_rmse']):
    axes[0].annotate(f'{tr:.3f}', (ep, tr), textcoords='offset points',
                     xytext=(0, 6), fontsize=7, ha='center', color='#2196F3')
    axes[0].annotate(f'{va:.3f}', (ep, va), textcoords='offset points',
                     xytext=(0,-12), fontsize=7, ha='center', color='#F44336')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('RMSE (normalised log µM)')
axes[0].set_title('LDO RMSE — overfitting from epoch 3\n(val 0.983→1.158, train continues down)')
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

models   = ['Ridge\nECFP4', 'Ridge\nOmics', 'RF', 'MLP', 'XGBoost', 'Bi-Int\n(ep.2)']
ldo_r    = [0.228,  0.228,  0.231, 0.225, 0.367, 0.316]
ci_low   = [0.196,  0.199,  0.202, 0.194, 0.338, 0.287]
ci_high  = [0.256,  0.258,  0.259, 0.255, 0.393, 0.344]
yerr     = [[r-l for r,l in zip(ldo_r,ci_low)], [h-r for h,r in zip(ci_high,ldo_r)]]
colors_b = ['#90CAF9','#B0BEC5','#B0BEC5','#B0BEC5','#EF9A9A','#A5D6A7']
bars = axes[1].bar(models, ldo_r, color=colors_b, edgecolor='k', lw=0.7,
                   yerr=yerr, capsize=4, error_kw={'linewidth': 1.2})
axes[1].axhline(0.367, color='red', linestyle='--', lw=1.5, label='XGBoost (best, r=0.367)')
axes[1].bar_label(bars, fmt='%.3f', padding=3, fontsize=9)
axes[1].set_ylabel('Pearson r (LDO validation)')
axes[1].set_title('LDO: Bi-Int vs classical baselines\n(with 95% bootstrap CI)')
axes[1].legend(fontsize=8); axes[1].set_ylim(0, 0.5); axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'nb_04_qsar_ldo.png'), dpi=110, bbox_inches='tight')
plt.show()
print('XGBoost r=0.367 vs Bi-Int r=0.316 (−0.051). Deep learning not yet beneficial on LDO.')
print('Figure saved → figures/nb_04_qsar_ldo.png')

---
## 6. Baseline comparison — all models × 3 splits

Five classical baselines were trained on the same 20k subsampled CCLE data:
- **Ridge** with ECFP4 (2048 bits) + omics features (4,452 dims total)
- **Ridge** with omics only
- **Random Forest** (50 trees, ECFP4+omics)
- **MLP** (256→128, ECFP4+omics)
- **XGBoost** (100 trees, ECFP4+omics)

**Key insight:** Ridge ECFP4+omics in LDO (r=0.228) is barely better than Ridge
omics-only (r=0.228), showing that ECFP4 adds noise for unseen drug scaffolds.
XGBoost is the most robust model across all three splits.

In [ ]:
if os.path.exists(BL_CSV):
    df_bl = pd.read_csv(BL_CSV)
    print(f'Loaded {len(df_bl)} rows from {BL_CSV}')
    print(df_bl[['Model','Split','Pearson_r','CI_low','CI_high','RMSE']].to_string(index=False))
else:
    print('Baseline CSV not found — using embedded values')
    df_bl = pd.DataFrame({
        'Model': ['Ridge (ECFP4+omics)']*3 + ['Ridge (omics only)']*3 +
                 ['RF (50 trees)']*3 + ['MLP (256→128)']*3,
        'Split': ['Random','Leave-Drug-Out','Leave-Cell-Out']*4,
        'Pearson_r': [0.230, 0.228, 0.183,
                      0.230, 0.228, 0.182,
                      0.240, 0.231, 0.200,
                      0.227, 0.225, 0.211],
    })

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Baselines vs Bi-Int — Pearson r by split', fontweight='bold', fontsize=12)

splits = ['Random', 'Leave-Drug-Out', 'Leave-Cell-Out']
titles = ['Random split\n(memorisation artefact — inflated)',
          'Leave-Drug-Out\n(honest: unseen drugs)',
          'Leave-Cell-Out\n(honest: unseen cell lines)']

# Add Bi-Int rows
biint = pd.DataFrame({'Model':['Bi-Int'],'Split':['Random'],'Pearson_r':[0.811]})
biint_ldo = pd.DataFrame({'Model':['Bi-Int'],'Split':['Leave-Drug-Out'],'Pearson_r':[0.316]})
df_all = pd.concat([df_bl, biint, biint_ldo], ignore_index=True)

for ax, split, title in zip(axes, splits, titles):
    sub = df_all[df_all['Split']==split].sort_values('Pearson_r', ascending=False)
    colors = ['#FF7043' if 'XGBoost' in m else
              '#A5D6A7' if 'Bi-Int' in m else '#90CAF9'
              for m in sub['Model']]
    bars = ax.bar(range(len(sub)), sub['Pearson_r'], color=colors, edgecolor='k', lw=0.6)
    ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=8)
    ax.set_xticks(range(len(sub)))
    ax.set_xticklabels([m.replace(' (ECFP4+omics)','').replace(' (omics only)','\n(omics)')
                        for m in sub['Model']], rotation=20, ha='right', fontsize=8)
    ax.set_ylabel('Pearson r'); ax.set_title(title)
    ax.set_ylim(0, 1.05); ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'nb_05_baselines.png'), dpi=110, bbox_inches='tight')
plt.show()
print('Figure saved → figures/nb_05_baselines.png')

---
## 7. GraphGA candidates — structures & drug-likeness

GraphGA is a genetic algorithm that evolves a population of molecules guided by
a composite score: **QED** (drug-likeness) + Bi-Int **IC50** prediction.

10 top candidates are stored in `graphga_top_candidates.csv`.

> **Caveat:** IC50 component of the GA score uses the same limited predictor
> (LDO r=0.316). The QED component is reliable; the IC50 ranking is not.

In [ ]:
if not os.path.exists(GA_CSV):
    print(f'GraphGA CSV not found: {GA_CSV}')
    print('Run: python src/graphga_biint_optimizer.py')
else:
    df_ga = pd.read_csv(GA_CSV)
    print(f'Loaded {len(df_ga)} candidates')
    cols = [c for c in ['rank','smiles','qed','mw','logp','sa','composite','pains'] if c in df_ga.columns]
    print(df_ga[cols].to_string(index=False))

    if RDKIT_OK:
        mols, legends = [], []
        for _, row in df_ga.iterrows():
            mol = Chem.MolFromSmiles(row['smiles'])
            if mol:
                rdDepictor.Compute2DCoords(mol)
                mols.append(mol)
                legends.append(f"#{int(row['rank'])}  QED={row['qed']:.3f}\nMW={row['mw']:.0f}")
        img = Draw.MolsToGridImage(mols, molsPerRow=5, subImgSize=(360, 290),
                                   legends=legends, returnPNG=False)
        img.save(os.path.join(FIG_DIR, '01_molecular_structures.png'))
        display(img)
        print('Figure saved → figures/01_molecular_structures.png')

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle('GraphGA candidates — drug-likeness profile', fontweight='bold')
    labels_ga = [f"#{int(r)}" for r in df_ga['rank']]

    colors_q = ['gold' if q >= 0.9 else '#42A5F5' if q >= 0.8 else '#EF9A9A' for q in df_ga['qed']]
    bars = axes[0].bar(labels_ga, df_ga['qed'], color=colors_q, edgecolor='k', lw=0.6)
    axes[0].axhline(0.7, color='red', linestyle='--', lw=1.5, label='Drug-like ≥0.7')
    axes[0].bar_label(bars, fmt='%.3f', padding=2, fontsize=7)
    axes[0].set_ylabel('QED'); axes[0].legend(fontsize=8); axes[0].set_ylim(0, 1.1)
    axes[0].set_title(f'Drug-Likeness (QED)  mean={df_ga["qed"].mean():.3f}')
    axes[0].grid(alpha=0.3, axis='y')

    axes[1].bar(labels_ga, df_ga['mw'], color='#66BB6A', edgecolor='k', lw=0.6)
    axes[1].axhline(500, color='red', linestyle='--', lw=1.5, label='Lipinski MW≤500')
    axes[1].set_ylabel('MW (Da)'); axes[1].legend(fontsize=8)
    axes[1].set_title(f'Molecular weight  range {df_ga["mw"].min():.0f}–{df_ga["mw"].max():.0f} Da')
    axes[1].grid(alpha=0.3, axis='y')

    colors_l = ['#42A5F5' if l <= 5 else '#EF5350' for l in df_ga['logp']]
    axes[2].bar(labels_ga, df_ga['logp'], color=colors_l, edgecolor='k', lw=0.6)
    axes[2].axhline(5, color='red', linestyle='--', lw=1.5, label='Lipinski LogP≤5')
    axes[2].set_ylabel('LogP'); axes[2].legend(fontsize=8)
    axes[2].set_title(f'LogP  mean={df_ga["logp"].mean():.2f} (optimum 1–3)')
    axes[2].grid(alpha=0.3, axis='y')

    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, '04_qed_lipinski.png'), dpi=110, bbox_inches='tight')
    plt.show()
    print('Figure saved → figures/04_qed_lipinski.png')

---
## 8. BRICS-DQN reward progression (5,000 episodes)

The BRICS-DQN agent assembles molecules from BRICS fragments using Double DQN.
The reward function combines:
- Bi-Int IC50 prediction (lower is better, clipped)
- QED score
- Tanimoto diversity penalty (vs. previous top molecules)
- Lipinski hard penalty (−1.0 if any rule violated by >1)

**Results:** 5,000 episodes, best reward = 6.124, validity = **60.5%** (global).
No clear upward trend in reward — the agent has not converged yet.
Planned fix: add valence penalty to reduce invalid SMILES rate.

In [ ]:
if not os.path.exists(DQN_CSV):
    print(f'DQN CSV not found: {DQN_CSV}')
    print('Run: python src/brics_dqn_optimizer.py')
else:
    df_dqn     = pd.read_csv(DQN_CSV)
    valid_mask = df_dqn['valid'].astype(str).str.lower() == 'true'
    rewards    = df_dqn['reward'].values
    episodes   = df_dqn['episode'].values
    rolling    = pd.Series(rewards).rolling(50, min_periods=1).mean().values

    window    = 100
    n_bins    = len(rewards) // window
    bin_means = [rewards[i*window:(i+1)*window].mean() for i in range(n_bins)]
    bin_valid = [valid_mask.values[i*window:(i+1)*window].mean()*100 for i in range(n_bins)]
    bin_x     = [(i + 0.5) * window for i in range(n_bins)]

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    fig.suptitle(f'BRICS-DQN — {len(df_dqn):,} episodes (real data, 24 May 2026)', fontweight='bold')

    axes[0].scatter(episodes[valid_mask],  rewards[valid_mask],
                    s=1.5, alpha=0.2, color='#42A5F5', label='Valid')
    axes[0].scatter(episodes[~valid_mask], rewards[~valid_mask],
                    s=1.5, alpha=0.1, color='#EF5350', label='Invalid')
    axes[0].plot(episodes, rolling, 'k-', lw=1.8, label='Rolling mean (w=50)', zorder=5)
    best_idx = rewards.argmax()
    axes[0].annotate(f'Best: {rewards.max():.3f}',
                     xy=(episodes[best_idx], rewards[best_idx]),
                     xytext=(episodes[best_idx]+300, rewards[best_idx]-0.5),
                     arrowprops=dict(arrowstyle='->', color='red'), color='red', fontsize=8)
    axes[0].set_xlabel('Episode'); axes[0].set_ylabel('Reward')
    axes[0].set_title(f'Rewards + rolling mean\nbest={rewards.max():.3f}, mean={rewards.mean():.3f}')
    axes[0].legend(markerscale=4, fontsize=8); axes[0].grid(alpha=0.2)

    axes[1].plot(bin_x, bin_means, 'o-', color='#1565C0', lw=2, markersize=4)
    axes[1].axhline(np.mean(bin_means), color='red', linestyle='--', lw=1.2,
                    label=f'Global mean = {np.mean(bin_means):.2f}')
    axes[1].set_xlabel('Episode'); axes[1].set_ylabel('Mean reward / 100 ep')
    axes[1].set_title('Mean reward per 100-episode block\n(no clear upward trend → not converged)')
    axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

    axes[2].plot(bin_x, bin_valid, '-^', color='#2E7D32', lw=2, markersize=4)
    axes[2].fill_between(bin_x, bin_valid, alpha=0.15, color='green')
    axes[2].axhline(valid_mask.mean()*100, color='red', linestyle='--', lw=1.5,
                    label=f'Global: {valid_mask.mean()*100:.1f}%')
    axes[2].set_xlabel('Episode'); axes[2].set_ylabel('% Valid SMILES')
    axes[2].set_title('Validity / 100-episode block\n(60.5% global — stable but improvable)')
    axes[2].set_ylim(0, 105); axes[2].legend(fontsize=8); axes[2].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, 'nb_07_dqn_reward.png'), dpi=110, bbox_inches='tight')
    plt.savefig(os.path.join(FIG_DIR, '03_dqn_reward.png'), dpi=110, bbox_inches='tight')
    plt.show()
    print(f'Total: {len(df_dqn):,} ep | Valid: {valid_mask.sum():,} ({valid_mask.mean()*100:.1f}%)')
    print(f'Best reward: {rewards.max():.4f} (ep {episodes[rewards.argmax()]})')
    print('Figure saved → figures/nb_07_dqn_reward.png + figures/03_dqn_reward.png')

---
## 9. Chemical validation — Tanimoto, SA score, MedChem filters

Full validation was run by `scripts/molecular_validation.py` on 60 candidates
(10 GraphGA + 50 BRICS-DQN top-reward). Results stored in
`Dataset/molecular_validation_report.csv`.

**Summary:**
- 38/60 (63%) pass ALL MedChem filters (PAINS + Brenk + Lipinski + Veber)
- 58/60 have Tanimoto < 0.30 vs CCLE drugs → structurally novel
- 0/60 have SA > 6 → entire library is synthetically accessible
- Internal diversity = **0.90** (mean Tanimoto = 0.10 across all candidate pairs)

In [ ]:
if not os.path.exists(VALID_CSV):
    print(f'Validation report not found: {VALID_CSV}')
    print('Run: python scripts/molecular_validation.py')
else:
    df_val = pd.read_csv(VALID_CSV)
    print(f'Loaded {len(df_val)} candidates from {VALID_CSV}')

    # Summary statistics
    n = len(df_val)
    print(f'\nMedChem summary:')
    print(f'  Clean (all filters pass): {df_val["medchem_clean"].sum()} / {n}')
    print(f'  PAINS alerts: {df_val["pains_flag"].sum()}')
    print(f'  Brenk alerts: {df_val["brenk_flag"].sum()}')
    print(f'  Lipinski fail: {(~df_val["lipinski_pass"]).sum()}')
    print(f'  Veber fail:    {(~df_val["veber_pass"]).sum()}')
    print(f'  SA > 6:        {df_val["sa_flag_hard"].sum()}')
    print(f'\nTanimoto vs CCLE:')
    print(f'  > 0.7  (analogue):  {(df_val["max_tanimoto_ccle"]>0.7).sum()}')
    print(f'  0.4–0.6 (ideal):   {((df_val["max_tanimoto_ccle"]>=0.4)&(df_val["max_tanimoto_ccle"]<=0.6)).sum()}')
    print(f'  < 0.3  (novel):    {(df_val["max_tanimoto_ccle"]<0.3).sum()}')

    print(f'\nTop-5 by quality score (QED 30% + SA 25% + diversity 20% + medchem 25%):')
    cols = ['id','source','quality_score','qed_computed','sa_score','medchem_clean','max_tanimoto_ccle','closest_ccle_drug']
    print(df_val[cols].head(5).to_string(index=False))

    # Plot: distribution of SA scores and Tanimoto
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle('Chemical validation — 60 candidates (31 May 2026)', fontweight='bold')

    axes[0].hist(df_val['sa_score'], bins=15, color='#42A5F5', edgecolor='k', lw=0.5)
    axes[0].axvline(6, color='red', linestyle='--', lw=1.5, label='SA=6 (hard threshold)')
    axes[0].set_xlabel('SA score (1=easy, 10=hard)'); axes[0].set_ylabel('Count')
    axes[0].set_title(f'Synthetic Accessibility\nmean={df_val["sa_score"].mean():.2f}, max={df_val["sa_score"].max():.2f}')
    axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

    axes[1].hist(df_val['max_tanimoto_ccle'], bins=15, color='#FF9800', edgecolor='k', lw=0.5)
    axes[1].axvline(0.3, color='red', linestyle='--', lw=1.5, label='< 0.3 = structurally novel')
    axes[1].axvline(0.7, color='green', linestyle='--', lw=1.5, label='> 0.7 = close analogue')
    axes[1].set_xlabel('Max Tanimoto vs CCLE drugs'); axes[1].set_ylabel('Count')
    axes[1].set_title(f'Tanimoto vs CCLE\n58/60 < 0.30 (structurally novel)')
    axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, 'nb_09_validation.png'), dpi=110, bbox_inches='tight')
    plt.show()
    print('Figure saved → figures/nb_09_validation.png')

---
## 10. Conclusions & limitations

### What works
- **GNN pre-training on ChEMBL** produces chemically meaningful embeddings (val RMSE 0.22 on 8 normalised descriptors). Transfer learning is effective.
- **Random split r = 0.811** confirms the model can fit the IC50 landscape when drug identity is available at training time.
- **60 drug-like candidates generated:** 63% pass all MedChem filters, internal diversity = 0.90, all synthetically accessible (SA ≤ 6).
- **Library is structurally original** vs CCLE reference drugs (58/60 Tanimoto < 0.30).

### What doesn't work yet
- **LDO r = 0.316 < XGBoost r = 0.367:** deep learning adds no benefit on 16k training triplets. Root causes: insufficient data, no early stopping, no SMILES augmentation.
- **BRICS-DQN validity 60.5%:** one in two generated molecules is chemically invalid. No convergence in 5,000 episodes.
- **65 drugs unmapped (24%):** reduces training coverage and LDO diversity.

### Next steps (prioritised)
1. Run LDO ablation (`scripts/ldo_ablation.py`): early stopping + dropout 0.3 + SMILES augmentation
2. Add valence penalty to BRICS-DQN reward to improve validity
3. Complete SMILES mapping for 65 missing drugs
4. In silico ADMET profiling (SwissADME, pkCSM) for top-3 candidates (BRI-46, BRI-12, BRI-58)
5. Molecular docking on HDAC1/6 and TrkA targets for promising scaffolds

### IC50 disclaimer
> ⚠️ **Predicted IC50 values for generated molecules are extrapolated by a model
> with limited leave-drug-out performance (r = 0.316).**
> These values must NOT be interpreted as reliable potency predictions.
> Biological validation (in vitro IC50 assay) is required before any conclusion
> about the pharmacological activity of these molecules.